In [ ]:
import numpy as np
from case_studies import *


def backtracking_line_search(f, df, x, p, alpha=1.0, rho=0.5, c1=1e-4):
    while f(x + alpha * p) > f(x) + c1 * alpha * np.dot(df(x), p):
        alpha *= rho
    return alpha

def solve_linear_system(B, A, grad_f):
    m, n = A.shape
    zero_block = np.zeros((m, m))
    lhs = np.block([[B, A.T], [A, zero_block]])
    rhs = np.concatenate([-grad_f, np.zeros(m)])
    
    solution = np.linalg.solve(lhs, rhs)
    
    p_k = solution[:n]  # Extract p_k
    lambda_k = solution[n:]  # Extract λ_k
    
    return p_k, lambda_k


# Newton's Method function (updated version)
def newtons_method(f, df, Hf, x0, max_iter=10000, epsilon=1e-9):
    x = x0.copy()
    xs = []
    grad_norms = []
    for _ in range(max_iter):
        grad = df(x)
        grad_norm = np.linalg.norm(grad)
        grad_norms.append(grad_norm)
        
        if grad_norm < epsilon:
            break
        
        H = Hf(x)
        eigvals, eigvecs = np.linalg.eigh(H)  

        if eigvals.min() > 0:  # Hessian is positive definite
            p = -np.linalg.solve(H, grad)
        else:
            new_H = sum((1 / abs(l)) * np.outer(v, v) for l, v in zip(eigvals, eigvecs.T))
            p = -new_H @ grad
        
        alpha = backtracking_line_search(f, df, x, p, 1.0)  
        x = x + alpha * p
        xs.append(x)
        
    return np.array(xs), np.array(grad_norms)


def constrained_newton(f, df, Hf, A, b, x0, tol=1e-9, max_iter=1000, alpha_init=1.0, rho=0.5, c1=1e-4):
    x_k = x0.copy()  
    xs = []  
    grad_norms = []  

    for k in range(max_iter):
        grad = df(x_k)
        grad_norm = np.linalg.norm(grad)
        grad_norms.append(grad_norm)
        
        if grad_norm < tol:  
            break
        
        H = Hf(x_k)
        
        if np.linalg.eigvals(H).min() > 0:  # Check if Hessian is positive definite
            B = H
        else:
            eigvals, eigvecs = np.linalg.eigh(H)
            B = sum(np.abs(l) * np.outer(v, v) for l, v in zip(eigvals, eigvecs))
  
        p_k, lambda_k = solve_linear_system(B, A, grad)

        alpha_k = backtracking_line_search(f, df, x_k, p_k, alpha_init, rho, c1)

        x_k = x_k + alpha_k * p_k
        xs.append(x_k.copy()) 
        
    return np.array(xs), np.array(grad_norms)


def constrained_steepest_descent(f, df, A, b, x0, tol=1e-6, max_iter=1000, c1=1e-4, rho=0.5):
    x_k = x0.copy()
    xs = [] 
    grad_norms = []  
    m, n = A.shape

    M = np.eye(n) - A.T @ np.linalg.inv(A @ A.T) @ A
    beta_k = 1 
    
    for k in range(max_iter):
        grad = df(x_k)
        grad_norm = np.linalg.norm(grad)
        grad_norms.append(grad_norm)
        
        if grad_norm < tol:
            break
        
        p_k = -M @ grad
        alpha_k = backtracking_line_search(f, df, x_k, p_k, beta_k, c1, rho)
        x_k = x_k + alpha_k * p_k
        xs.append(x_k.copy())
        beta_k = alpha_k / rho
    
    return np.array(xs), np.array(grad_norms)

In [ ]:
def testPrint():
    # Dimensions for each function
    n1, n2, n3, n4, n5 = 5, 2, 5, 5, 5
    
    # Constraint matrices
    # Change based on which constraints we want to test
    A1 = np.random.randn(2, n1)  # 2 constraints for f1
    A2 = np.random.randn(1, n2)  # 1 constraint for f2
    A3 = np.random.randn(3, n3)  # 3 constraints for f3
    A4 = np.random.randn(1, n4)  # 1 constraint for f4
    A5 = np.random.randn(2, n5)  # 2 constraints for f5
    
    # Random b vectors
    b1 = np.random.randn(2)
    b2 = np.random.randn(1)
    b3 = np.random.randn(3)
    b4 = np.random.randn(1)
    b5 = np.random.randn(2)
    
    # Generate feasible initial points that satisfy Ax + b = 0
    x0_f1 = np.linalg.lstsq(A1, -b1, rcond=None)[0]
    x0_f2 = np.linalg.lstsq(A2, -b2, rcond=None)[0]
    x0_f3 = np.linalg.lstsq(A3, -b3, rcond=None)[0]
    x0_f4 = np.linalg.lstsq(A4, -b4, rcond=None)[0]
    x0_f5 = np.linalg.lstsq(A5, -b5, rcond=None)[0]

    # Running Newton's Method
    print("\nNewton's Method Results:")
    print("f1:", newtons_method(f1, df1, Hf1, x0_f1)[0])
    print("f2:", newtons_method(f2, df2, Hf2, x0_f2)[0])
    print("f3:", newtons_method(f3, df3, Hf3, x0_f3)[0])
    print("f4:", newtons_method(f4, df4, Hf4, x0_f4)[0])
    print("f5:", newtons_method(f5, df5, Hf5, x0_f5)[0])

    # Running Constrained Newton's Method
    print("\nConstrained Newton's Method Results:")
    print("f1:", constrained_newton(f1, df1, Hf1, A1, b1, x0_f1))
    print("f2:", constrained_newton(f2, df2, Hf2, A2, b2, x0_f2))
    print("f3:", constrained_newton(f3, df3, Hf3, A3, b3, x0_f3))
    print("f4:", constrained_newton(f4, df4, Hf4, A4, b4, x0_f4))
    print("f5:", constrained_newton(f5, df5, Hf5, A5, b5, x0_f5))

    # Running Constrained Steepest Descent
    print("\nConstrained Steepest Descent Results:")
    print("f1:", constrained_steepest_descent(f1, df1, A1, b1, x0_f1))
    print("f2:", constrained_steepest_descent(f2, df2, A2, b2, x0_f2))
    print("f3:", constrained_steepest_descent(f3, df3, A3, b3, x0_f3))
    print("f4:", constrained_steepest_descent(f4, df4, A4, b4, x0_f4))
    print("f5:", constrained_steepest_descent(f5, df5, A5, b5, x0_f5))
    
testPrint()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def project_onto_affine_subspace(A, b, x_init):
    """
    Project the initial guess x_init onto the affine subspace defined by Ax + b = 0.
    """
    A_pseudo_inv = np.linalg.pinv(A)
    x_proj = x_init - A_pseudo_inv @ (A @ x_init + b)
    
    # Optionally check that the projection is indeed satisfying the constraint within tolerance
    assert np.allclose(A @ x_proj + b, 0, atol=1e-9), "Projection did not satisfy Ax + b = 0"
    
    return x_proj


def generate_full_rank_A(m, n):
    A = np.random.randn(m, n) 
    Q, R = np.linalg.qr(A)
    return Q

In [ ]:
def test_optimizers_on_function_distance(functions, optimizers, A, b, dimension, alpha_init=1.0, rho=0.5, c1=1e-4, c2=0.9, epsilon=1e-9, max_iter=1000, num_runs=5):
    """
    Test optimizers on a given set of functions and plot ||x_k - x*|| vs Iterations.
    Runs multiple times with different starting points and plots each individually.

    Parameters:
    - functions: List of tuples (function, gradient, Hessian)
    - optimizers: List of optimizers (functions)
    - A: Constraint matrix
    - b: Constraint vector
    - dimension: Dimension of the problem (default is 20)
    - epsilon: Tolerance for stopping condition (default is 1e-9)
    - max_iter: Maximum number of iterations (default is 1000)
    - num_runs: Number of runs with different initializations (default is 5)
    """
    colors = ['b', 'r', 'g', 'orange', 'purple']  # Assign unique colors for each run

    for f, df, Hf in functions:
        plt.figure(figsize=(6, 4))
        plt.title(f"{f.__name__} (d={dimension}, ε={epsilon:.0e}, α_init={alpha_init}, ρ={rho}, c1={c1})")

        for opt_idx, opt in enumerate(optimizers):
            color = colors[opt_idx % len(colors)]  # Assign color based on optimizer index

            for run in range(num_runs):
                x0 = np.linalg.lstsq(A, -b, rcond=None)[0]
                
                # Check if x0 satisfies the constraint A x0 + b = 0 (within a small tolerance)
                if not np.allclose(A @ x0 + b, 0):
                # If x0 doesn't satisfy the constraint, project it onto the affine subspace
                    x0 = project_onto_affine_subspace(A, b, x0)
        

                if opt == constrained_newton:
                    xs,_ = opt(f, df, Hf, A, b, x0, tol=epsilon, max_iter=max_iter, alpha_init=alpha_init, rho=rho, c1=c1)
                elif opt == constrained_steepest_descent:
                    xs,_ = opt(f, df, A, b, x0, tol=epsilon, max_iter=max_iter, c1=c1, rho=rho)

                optimal = x_opt(f.__name__, dimension)
                distances = [np.linalg.norm(x - optimal) for x in xs]

                label = f"{opt.__name__}" if run == 0 else ""
                plt.plot(range(len(distances)), distances, linestyle='--', color=color, alpha=0.8, label=label)

        plt.xlabel("Iterations")
        plt.ylabel(r"$||x_k - x^*||$")
        plt.yscale("log") 
        plt.grid()
        plt.legend()
        plt.show()

In [ ]:
m,n = 3,5 # m should <= n
A =  np.random.randn(m, n)  # Constraint matrix (m linear constraints)
b = np.random.randn(m)  # Constraint vector

functions = [(f1, df1, Hf1), (f3, df3, Hf3), (f4, df4, Hf5), (f5, df5, Hf5)]

optimizers = [constrained_newton, constrained_steepest_descent]

test_optimizers_on_function_distance(functions, optimizers, A, b, dimension=n, alpha_init=1.0, rho=0.5, c1=1e-4, c2=0.9, epsilon=1e-9, max_iter=1000, num_runs=1)

In [ ]:
m,n = 2,2 # m should <= n
A =  np.random.randn(m, n)  # Constraint matrix (m linear constraints)
b = np.random.randn(m)  # Constraint vector

functions = [(f2, df2, Hf2)]

optimizers = [constrained_newton, constrained_steepest_descent]

test_optimizers_on_function_distance(functions, optimizers, A, b, dimension=n, alpha_init=1.0, rho=0.5, c1=1e-4, c2=0.9, epsilon=1e-9, max_iter=1000, num_runs=1)